# Image Denoising with Autoencoder Architectures

**Dataset:** MNIST handwritten digits (28x28, grayscale)

**Objective:** Add synthetic noise to clean digit images, then train and
compare three autoencoder variants that try to recover the original,
noise-free images.

**Architectures built**
1. Fully-connected (dense) autoencoder
2. CNN autoencoder using `Conv2DTranspose` for upsampling
3. CNN autoencoder using `UpSampling2D` + `Conv2D` for upsampling

**Evaluation:** reconstruction loss (binary cross-entropy), MSE, and PSNR
on the test split, plus a noise-robustness sweep at the end.


In [ ]:
# ----- Library imports -----
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Conv2D, MaxPooling2D,
    Conv2DTranspose, UpSampling2D, Rescaling
)
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ----- Fix random seeds for repeatable results -----
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("Running on TensorFlow", tf.__version__)


In [ ]:
# ----- Read MNIST images from folder structure -----
DATASET_DIR = "dataset/mnist"
TARGET_SIZE = (28, 28)
BATCH_SIZE = 128

train_data = image_dataset_from_directory(
    os.path.join(DATASET_DIR, "train"),
    labels="inferred",
    label_mode="int",
    color_mode="grayscale",
    image_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_STATE,
)

test_data = image_dataset_from_directory(
    os.path.join(DATASET_DIR, "test"),
    labels="inferred",
    label_mode="int",
    color_mode="grayscale",
    image_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

normalize = Rescaling(1.0 / 255)
train_data = train_data.map(lambda img, lbl: (normalize(img), lbl))
test_data = test_data.map(lambda img, lbl: (normalize(img), lbl))

# Materialize the whole dataset as numpy arrays for easy slicing later
X_train = np.concatenate([batch.numpy() for batch, _ in train_data], axis=0)
X_test = np.concatenate([batch.numpy() for batch, _ in test_data], axis=0)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)


In [ ]:
# ----- Quick visual sanity check on the raw digits -----
fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))
for ax, img in zip(axes, X_train[:8]):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.axis("off")
fig.suptitle("Raw MNIST samples (scaled to [0, 1])")
plt.tight_layout()
plt.show()


In [ ]:
# ----- Cache the preprocessed tensors to disk -----
CACHE_DIR = "cache"
os.makedirs(os.path.join(CACHE_DIR, "train"), exist_ok=True)
os.makedirs(os.path.join(CACHE_DIR, "test"), exist_ok=True)

np.save(os.path.join(CACHE_DIR, "train", "X_train.npy"), X_train)
np.save(os.path.join(CACHE_DIR, "test", "X_test.npy"), X_test)
print("Cached preprocessed arrays under:", CACHE_DIR)


In [ ]:
# ----- Inject Gaussian noise to build the (noisy, clean) training pairs -----
NOISE_LEVEL = 0.5

def add_gaussian_noise(images, level=NOISE_LEVEL, seed=RANDOM_STATE):
    generator = np.random.default_rng(seed)
    corrupted = images + level * generator.normal(size=images.shape)
    return np.clip(corrupted, 0.0, 1.0)

X_train_noisy = add_gaussian_noise(X_train, seed=RANDOM_STATE)
X_test_noisy = add_gaussian_noise(X_test, seed=RANDOM_STATE + 1)

print(f"Applied Gaussian noise with sigma={NOISE_LEVEL}")


In [ ]:
# ----- Visual comparison: clean vs corrupted -----
n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(14, 3.2))
for col in range(n_show):
    axes[0, col].imshow(X_train[col].squeeze(), cmap="gray")
    axes[1, col].imshow(X_train_noisy[col].squeeze(), cmap="gray")
    axes[0, col].axis("off")
    axes[1, col].axis("off")
axes[0, 0].set_ylabel("clean")
axes[1, 0].set_ylabel("noisy")
plt.tight_layout()
plt.show()


In [ ]:
# ----- Utility functions shared across all three models -----

def compute_mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))

def compute_psnr(y_true, y_pred):
    return float(tf.reduce_mean(tf.image.psnr(y_true, y_pred, max_val=1.0)))

def train_autoencoder(model, noisy_input, clean_target, weights_path, max_epochs=20):
    model.compile(optimizer="adam", loss="binary_crossentropy")
    early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    checkpoint = ModelCheckpoint(weights_path, monitor="val_loss", save_best_only=True)
    return model.fit(
        noisy_input, clean_target,
        epochs=max_epochs,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        callbacks=[early_stop, checkpoint],
        verbose=1,
    )

def plot_training_curve(history, title, line_color="steelblue"):
    plt.figure(figsize=(7, 4))
    plt.plot(history.history["loss"], color=line_color, label="training")
    plt.plot(history.history["val_loss"], color="crimson", linestyle="--", label="validation")
    plt.title(title)
    plt.xlabel("epoch")
    plt.ylabel("binary cross-entropy")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

def show_reconstruction_grid(clean, noisy, reconstructed, count=8, cmap="gray"):
    labeled_rows = [("clean", clean), ("noisy", noisy), ("reconstructed", reconstructed)]
    plt.figure(figsize=(14, 5))
    for row_idx, (row_label, row_data) in enumerate(labeled_rows):
        for col_idx in range(count):
            ax = plt.subplot(3, count, row_idx * count + col_idx + 1)
            plt.imshow(row_data[col_idx].squeeze(), cmap=cmap)
            plt.axis("off")
            if col_idx == 0:
                ax.set_title(row_label, loc="left", fontsize=10)
    plt.tight_layout()
    plt.show()

os.makedirs("checkpoints", exist_ok=True)
scoreboard = {}   # will hold metrics for every model we train


## Autoencoder #1 — Fully-Connected (Dense)

The simplest possible design: flatten each 28x28 image into a 784-length
vector, compress it down to a 64-dimensional code, then reconstruct it back
out. Because flattening throws away the 2D layout, this serves as our
baseline for the two convolutional models that follow.


In [ ]:
# ----- Flatten images for the dense network -----
X_train_flat = X_train.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)
X_train_noisy_flat = X_train_noisy.reshape(-1, 784)
X_test_noisy_flat = X_test_noisy.reshape(-1, 784)

def build_dense_autoencoder(input_dim=784, latent_dim=64):
    inputs = Input(shape=(input_dim,))
    h = Dense(512, activation="relu")(inputs)
    h = Dense(256, activation="relu")(h)
    latent = Dense(latent_dim, activation="relu")(h)
    h = Dense(256, activation="relu")(latent)
    h = Dense(512, activation="relu")(h)
    outputs = Dense(input_dim, activation="sigmoid")(h)
    return Model(inputs, outputs, name="dense_autoencoder")

dense_model = build_dense_autoencoder()
dense_model.summary()


In [ ]:
dense_history = train_autoencoder(
    dense_model, X_train_noisy_flat, X_train_flat,
    "checkpoints/dense_autoencoder.keras"
)
plot_training_curve(dense_history, "Dense Autoencoder — Training Loss")


In [ ]:
# ----- Evaluate the dense model on the test set -----
dense_loss = dense_model.evaluate(X_test_noisy_flat, X_test_flat, verbose=0)
dense_recon = dense_model.predict(X_test_noisy_flat).reshape(-1, 28, 28, 1)

show_reconstruction_grid(X_test, X_test_noisy, dense_recon)

scoreboard["Dense Autoencoder"] = {
    "loss": dense_loss,
    "mse": compute_mse(X_test, dense_recon),
    "psnr": compute_psnr(X_test, dense_recon),
}
dense_model.save("checkpoints/dense_autoencoder_final.keras")
print(scoreboard["Dense Autoencoder"])


## Autoencoder #2 — Convolutional, `Conv2DTranspose` Decoder

This version keeps the image in its native 2D shape throughout. The encoder
uses stacked `Conv2D` + `MaxPooling2D` blocks, and the decoder mirrors that
with learned `Conv2DTranspose` layers to upsample back to the original
resolution.


In [ ]:
def build_conv_autoencoder(upsampling_mode="transpose"):
    inputs = Input(shape=(28, 28, 1))

    # ---- encoder (identical for both decoder variants) ----
    e = Conv2D(32, 3, activation="relu", padding="same")(inputs)
    e = MaxPooling2D(2, padding="same")(e)
    e = Conv2D(64, 3, activation="relu", padding="same")(e)
    latent = MaxPooling2D(2, padding="same")(e)

    # ---- decoder ----
    if upsampling_mode == "transpose":
        d = Conv2DTranspose(64, 3, strides=2, activation="relu", padding="same")(latent)
        d = Conv2DTranspose(32, 3, strides=2, activation="relu", padding="same")(d)
        model_name = "conv_transpose_autoencoder"
    else:
        d = Conv2D(64, 3, activation="relu", padding="same")(latent)
        d = UpSampling2D(2)(d)
        d = Conv2D(32, 3, activation="relu", padding="same")(d)
        d = UpSampling2D(2)(d)
        model_name = "conv_upsampling_autoencoder"

    outputs = Conv2D(1, 3, activation="sigmoid", padding="same")(d)
    return Model(inputs, outputs, name=model_name)

convT_model = build_conv_autoencoder("transpose")
convT_model.summary()


In [ ]:
convT_history = train_autoencoder(
    convT_model, X_train_noisy, X_train,
    "checkpoints/conv_transpose_autoencoder.keras"
)
plot_training_curve(convT_history, "Conv2DTranspose Autoencoder — Training Loss", line_color="darkorange")


In [ ]:
convT_loss = convT_model.evaluate(X_test_noisy, X_test, verbose=0)
convT_recon = convT_model.predict(X_test_noisy)

show_reconstruction_grid(X_test, X_test_noisy, convT_recon)

scoreboard["Conv2DTranspose Autoencoder"] = {
    "loss": convT_loss,
    "mse": compute_mse(X_test, convT_recon),
    "psnr": compute_psnr(X_test, convT_recon),
}
convT_model.save("checkpoints/conv_transpose_autoencoder_final.keras")
print(scoreboard["Conv2DTranspose Autoencoder"])


## Autoencoder #3 — Convolutional, `UpSampling2D` Decoder

Same encoder as Autoencoder #2, but the decoder swaps transposed
convolutions for nearest-neighbour `UpSampling2D` followed by regular
`Conv2D` layers. This tends to avoid the checkerboard artifacts that
transposed convolutions can introduce, at a slightly lower parameter cost.


In [ ]:
convU_model = build_conv_autoencoder("upsample")
convU_model.summary()


In [ ]:
convU_history = train_autoencoder(
    convU_model, X_train_noisy, X_train,
    "checkpoints/conv_upsampling_autoencoder.keras"
)
plot_training_curve(convU_history, "UpSampling2D Autoencoder — Training Loss", line_color="seagreen")


In [ ]:
convU_loss = convU_model.evaluate(X_test_noisy, X_test, verbose=0)
convU_recon = convU_model.predict(X_test_noisy)

show_reconstruction_grid(X_test, X_test_noisy, convU_recon)

scoreboard["UpSampling2D Autoencoder"] = {
    "loss": convU_loss,
    "mse": compute_mse(X_test, convU_recon),
    "psnr": compute_psnr(X_test, convU_recon),
}
convU_model.save("checkpoints/conv_upsampling_autoencoder_final.keras")
print(scoreboard["UpSampling2D Autoencoder"])


## Side-by-Side Comparison

Gathering all three scoreboards into a single table and plotting each
metric for a quick visual comparison.


In [ ]:
results_table = pd.DataFrame(scoreboard).T.reset_index().rename(columns={"index": "Model"})
results_table.to_csv("checkpoints/model_comparison.csv", index=False)
results_table


In [ ]:
bar_colors = ["#5B84B1", "#E08E45", "#4E9F50"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric_name, y_label in zip(
    axes,
    ["loss", "mse", "psnr"],
    ["Binary Cross-Entropy", "Mean Squared Error", "PSNR (dB)"],
):
    ax.bar(results_table["Model"], results_table[metric_name], color=bar_colors)
    ax.set_ylabel(y_label)
    ax.set_xticklabels(results_table["Model"], rotation=20, ha="right")
fig.suptitle("Autoencoder Comparison")
plt.tight_layout()
plt.show()


In [ ]:
# ----- Compare all three models on one example digit -----
sample_idx = 3
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
panels = [
    ("original", X_test[sample_idx]),
    ("noisy", X_test_noisy[sample_idx]),
    ("dense", dense_recon[sample_idx]),
    ("conv-transpose", convT_recon[sample_idx]),
    ("conv-upsample", convU_recon[sample_idx]),
]
for ax, (label, img) in zip(axes, panels):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout()
plt.show()

top_model = results_table.loc[results_table["mse"].idxmin()]
print("Best-performing model (lowest MSE):\n", top_model)


In [ ]:
# ----- Robustness check: does the winning model hold up under heavier noise? -----
model_lookup = {
    "Dense Autoencoder": dense_model,
    "Conv2DTranspose Autoencoder": convT_model,
    "UpSampling2D Autoencoder": convU_model,
}
winner = model_lookup[top_model["Model"]]

for noise_level in [0.2, 0.4, 0.6, 0.8]:
    noisy_probe = add_gaussian_noise(X_test, level=noise_level, seed=123)
    probe_input = noisy_probe.reshape(-1, 784) if top_model["Model"] == "Dense Autoencoder" else noisy_probe
    probe_output = winner.predict(probe_input, verbose=0)
    if top_model["Model"] == "Dense Autoencoder":
        probe_output = probe_output.reshape(-1, 28, 28, 1)
    print(f"noise level={noise_level:>4} -> MSE={compute_mse(X_test, probe_output):.5f}")


## Observations

- **Data pipeline:** MNIST images were loaded from a labeled folder
  structure, rescaled into `[0, 1]`, and cached to disk as numpy arrays;
  Gaussian noise (`sigma = 0.5`) was then added to produce noisy/clean
  training pairs.
- **Dense autoencoder:** captures the overall digit shape but flattening
  discards spatial structure, so outputs come out somewhat blurred at the
  edges.
- **Conv2DTranspose autoencoder:** preserving the 2D layout through
  convolutions yields visibly crisper strokes and a lower reconstruction
  error than the dense baseline.
- **UpSampling2D autoencoder:** performs about as well as the transpose
  variant, giving up a bit of sharpness in exchange for smoother edges
  free of the checkerboard artifacts transposed convolutions sometimes
  produce.
- **Robustness sweep:** reconstruction error increases with the noise
  level for all three architectures, as expected, though the two
  convolutional models degrade more gracefully than the dense one at
  higher noise levels.


## Conclusion

All three autoencoders were able to remove Gaussian noise from MNIST
digits with varying levels of success. The dense network trains fastest
and is the simplest of the three, but it plateaus at blurrier
reconstructions since it throws away spatial information during
flattening. Both convolutional variants retain that spatial structure and
outperform the dense baseline on MSE and PSNR — the `Conv2DTranspose`
decoder achieves the sharpest reconstructions overall, while `UpSampling2D`
delivers comparable quality with smoother, artifact-free output at a lower
parameter cost.

**Possible next steps:** try a variational autoencoder (VAE), add residual
or U-Net-style skip connections, experiment with other noise types
(salt-and-pepper, speckle), and repeat the same pipeline on a harder
dataset such as CIFAR-10.
